# Advanced Weather Trend Forecasting

This notebook supports the Streamlit dashboard with a reproducible analysis workflow for the PM Accelerator advanced data science assessment.

It covers data cleaning, exploratory analysis, leakage-aware forecasting, anomaly detection, feature importance, and air-quality relationships.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import load_and_clean, location_daily
from src.forecasting import evaluate_models, future_forecast
from src.anomaly_detection import detect_anomalies
from src.feature_importance import importance_tables

DATA = Path("../data/GlobalWeatherRepository.csv")
df = load_and_clean(DATA)
df.shape

In [ ]:
df[["last_updated","temperature_celsius","precip_mm","humidity","air_quality_PM2.5"]].describe(include="all")

In [ ]:
country, location = "United Kingdom", "London"
daily = location_daily(df, country, location)
daily.head(), daily.shape

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(daily["date"], daily["temperature_celsius"], alpha=.5, label="Daily")
ax.plot(daily["date"], daily["temperature_celsius"].rolling(30).mean(), label="30-day mean")
ax.set(title=f"Temperature trend — {location}", ylabel="°C")
ax.legend()
plt.show()

In [ ]:
corr_cols = ["temperature_celsius","humidity","wind_kph","pressure_mb","precip_mm","cloud","visibility_km","uv_index","air_quality_PM2.5","air_quality_PM10"]
corr = daily[corr_cols].corr()
fig, ax = plt.subplots(figsize=(10,7))
sns.heatmap(corr, cmap="vlag", center=0, ax=ax)
ax.set_title("Weather and air-quality correlations")
plt.show()

In [ ]:
metrics, predictions, fitted, feature_names = evaluate_models(daily)
metrics

In [ ]:
future = future_forecast(daily, horizon=7)
future

In [ ]:
anomalies = detect_anomalies(daily)
anomalies[anomalies["anomaly"]].sort_values("anomaly_score", ascending=False).head(10)

In [ ]:
intrinsic, permutation = importance_tables(daily)
print("Random Forest importance")
display(intrinsic.head(10))
print("Permutation importance")
display(permutation.head(10))

## Interpretation notes

The test period is chronological rather than random, which prevents future observations from leaking into training. The dataset covers only about 2.4 years, so the analysis should be described as multi-year **weather** patterns rather than evidence of long-term climate change. Correlations between PM2.5 and weather variables are descriptive and do not establish causality.